# LeetCode #34 — Find First and Last Position of Element in Sorted Array: Lower Bound & Upper Bound

**Mức độ:** Medium · **Chủ đề:** Array, Binary Search

Skeleton trên LeetCode:

```python
class Solution:
    def searchRange(self, nums: List[int], target: int) -> List[int]:
```

Binary search thường chỉ trả lời "**có** `target` không, ở **một** vị trí nào đó". Bài này hỏi khó hơn: **vị trí đầu tiên** và **vị trí cuối cùng** của `target`, khi mảng có thể chứa nhiều phần tử bằng nhau.

Mục tiêu của notebook là nắm chắc hai biến thể quan trọng nhất của binary search:

- **Lower bound:** vị trí đầu tiên có `nums[i] >= x`.
- **Upper bound:** vị trí đầu tiên có `nums[i] > x`.

và module `bisect` có sẵn của Python.

# 1. Bài toán là gì?

Cho mảng `nums` sắp xếp **không giảm** (có thể có phần tử trùng) và `target`. Tìm **vị trí bắt đầu và kết thúc** của `target`. Nếu không có, trả `[-1, -1]`.

**Yêu cầu `O(log n)`.**

**Constraints (theo đề):**

- `0 <= nums.length <= 10^5`
- `-10^9 <= nums[i], target <= 10^9`
- `nums` không giảm.

---

## Ví dụ

```text
nums = [5, 7, 7, 8, 8, 10], target = 8  ->  [3, 4]
nums = [5, 7, 7, 8, 8, 10], target = 6  ->  [-1, -1]
nums = [],                  target = 0  ->  [-1, -1]
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Tìm kiếm nhị phân (binary search)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Tìm kiếm nhị phân (binary search)

### Ý tưởng

Trên mảng **đã sắp xếp**, so phần tử cần tìm với phần tử **ở giữa**: nếu nhỏ hơn thì chắc chắn nằm ở nửa trái, lớn hơn thì ở nửa phải. Mỗi bước loại **một nửa** → chỉ `O(log n)` bước (mảng 1 triệu phần tử: ~20 bước).

```text
tìm 23 trong [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
lo=0 hi=9 mid=4 → 16 < 23 → lo = 5
lo=5 hi=9 mid=7 → 56 > 23 → hi = 6
lo=5 hi=6 mid=5 → 23 = 23 → tìm thấy ở chỉ số 5
```

In [1]:
def tim_nhi_phan(a, x):
    lo, hi = 0, len(a) - 1                # khoảng tìm đóng [lo, hi]
    while lo <= hi:
        mid = (lo + hi) // 2
        print(f"   lo={lo} hi={hi} mid={mid} a[mid]={a[mid]}")
        if a[mid] == x:
            return mid
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

a_bs = [2, 5, 8, 12, 16, 23, 38, 56, 72, 91]
print("→", tim_nhi_phan(a_bs, 23))
print("→", tim_nhi_phan(a_bs, 7), "(không có)")

   lo=0 hi=9 mid=4 a[mid]=16
   lo=5 hi=9 mid=7 a[mid]=56
   lo=5 hi=6 mid=5 a[mid]=23
→ 5
   lo=0 hi=9 mid=4 a[mid]=16
   lo=0 hi=3 mid=1 a[mid]=5
   lo=2 hi=3 mid=2 a[mid]=8
→ -1 (không có)


### Dạng tổng quát: tìm vị trí **đầu tiên** thỏa điều kiện

Rất nhiều bài không hỏi "có `x` không" mà hỏi "vị trí đầu tiên mà điều kiện `P(i)` đúng", với `P` **đơn điệu**: `False False … False True True … True`. Khung:

```text
lo, hi = 0, n            # đáp án nằm trong [lo, hi]; n nghĩa là "không có"
while lo < hi:
    mid = (lo + hi) // 2
    if P(mid): hi = mid          # mid có thể là đáp án → giữ lại
    else:      lo = mid + 1      # mid chắc chắn không phải
return lo
```

Hai trường hợp quan trọng:

- **lower_bound**: vị trí đầu tiên `a[i] >= x` (`bisect_left`).
- **upper_bound**: vị trí đầu tiên `a[i] > x` (`bisect_right`).

Số lần xuất hiện của `x` = `upper_bound − lower_bound`.

In [2]:
from bisect import bisect_left, bisect_right

def dau_tien(n, P):
    lo, hi = 0, n
    while lo < hi:
        mid = (lo + hi) // 2
        if P(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

b_bs = [1, 2, 2, 2, 5, 7]
lb = dau_tien(len(b_bs), lambda i: b_bs[i] >= 2)
ub = dau_tien(len(b_bs), lambda i: b_bs[i] > 2)
print("lower_bound(2) =", lb, "| upper_bound(2) =", ub, "| số lần xuất hiện:", ub - lb)
print("bisect_left / bisect_right:", bisect_left(b_bs, 2), bisect_right(b_bs, 2))
print("chèn 6 vào vị trí:", bisect_left(b_bs, 6))
print("số nguyên nhỏ nhất k với k*k >= 50:", dau_tien(100, lambda k: k * k >= 50))

lower_bound(2) = 1 | upper_bound(2) = 4 | số lần xuất hiện: 3
bisect_left / bisect_right: 1 4
chèn 6 vào vị trí: 5
số nguyên nhỏ nhất k với k*k >= 50: 8


### Lỗi hay gặp (nguồn lỗi số 1 là "lệch 1")

- Trộn hai kiểu khoảng: `while lo <= hi` đi với `hi = mid - 1`; `while lo < hi` đi với `hi = mid`. Trộn lẫn → **lặp vô hạn** hoặc bỏ sót.
- `lo = mid` (thay vì `mid + 1`) với `mid = (lo+hi)//2` → khi `hi = lo + 1` thì `mid = lo`, không tiến → lặp vô hạn.
- Áp dụng trên mảng **chưa sắp xếp** / điều kiện không đơn điệu.
- (Ngôn ngữ khác) `(lo + hi) / 2` tràn số → dùng `lo + (hi - lo) // 2`; Python không bị.

In [3]:
def sai_lo_bang_mid(a, x, gioi_han=20):
    lo, hi, buoc = 0, len(a) - 1, 0
    while lo < hi and buoc < gioi_han:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid            # SAI: không loại mid
        else:
            hi = mid
        buoc += 1
    return buoc
print("lo = mid: số vòng trước khi bị chặn:", sai_lo_bang_mid([1, 3], 3))

lo = mid: số vòng trước khi bị chặn: 20


## 2.2 Áp dụng vào bài này

Vị trí đầu tiên = **lower_bound** (vị trí đầu tiên `>= target`), vị trí cuối = **upper_bound − 1** (vị trí đầu tiên `> target`, lùi 1). Cả hai là dạng "tìm vị trí đầu tiên thỏa điều kiện đơn điệu" ở trên.

# 3. Cách dễ nghĩ nhất: duyệt tuyến tính

In [4]:
from typing import List

def search_range_linear(nums: List[int], target: int) -> List[int]:
    first = last = -1
    for i, x in enumerate(nums):
        if x == target:
            if first == -1:
                first = i
            last = i
    return [first, last]

print(search_range_linear([5, 7, 7, 8, 8, 10], 8))
print(search_range_linear([5, 7, 7, 8, 8, 10], 6))

[3, 4]
[-1, -1]


`O(n)` — không đạt yêu cầu.

## Cách "nửa vời": binary search tìm một vị trí rồi lan ra hai bên

Tìm **một** vị trí bằng binary search, rồi đi sang trái / phải tới khi hết `target`.

Vấn đề: nếu mảng là `[8, 8, 8, ..., 8]` (10^5 phần tử), bước lan ra tốn `O(n)`. Xấu nhất vẫn `O(n)`.

# 4. Ý tưởng: hai lần Binary Search "lệch"

Thay vì dừng khi gặp `target`, ta **tiếp tục thu hẹp** về phía cần tìm:

- Tìm **vị trí đầu tiên**: gặp `nums[mid] == target` → ghi nhận, nhưng **tiếp tục tìm bên trái**.
- Tìm **vị trí cuối cùng**: gặp `nums[mid] == target` → ghi nhận, nhưng **tiếp tục tìm bên phải**.

Mỗi lần vẫn `O(log n)`, tổng `O(log n)`.

In [5]:
def find_first(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    ans = -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            ans = mid
            hi = mid - 1          # còn có thể có target bên TRÁI
        elif nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return ans


def find_last(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums) - 1
    ans = -1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            ans = mid
            lo = mid + 1          # còn có thể có target bên PHẢI
        elif nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return ans

nums = [5, 7, 7, 8, 8, 10]
print(find_first(nums, 8), find_last(nums, 8))
print(find_first(nums, 6), find_last(nums, 6))

3 4
-1 -1


# 5. Góc nhìn tổng quát: Lower Bound và Upper Bound

Cách viết ở mục 4 dễ hiểu, nhưng có một cách nhìn **mạnh hơn** dùng được cho rất nhiều bài:

- `lower_bound(x)` = vị trí **đầu tiên** có `nums[i] >= x` (hoặc `n` nếu không có).
- `upper_bound(x)` = vị trí **đầu tiên** có `nums[i] > x` (hoặc `n`).

Với `nums = [5, 7, 7, 8, 8, 10]`:

| x | lower_bound | upper_bound | Giải thích |
|---|---|---|---|
| 8 | 3 | 5 | `8` nằm ở chỉ số 3..4 |
| 7 | 1 | 3 | `7` nằm ở 1..2 |
| 6 | 1 | 1 | không có `6`; đó là chỗ **chèn** `6` |
| 11 | 6 | 6 | lớn hơn mọi phần tử |

Vậy:

```text
first = lower_bound(target)
last  = upper_bound(target) - 1
target có mặt  <=>  first < n  và  nums[first] == target
```

## Hình dung: mảng là một dãy `False ... False True ... True`

Với `lower_bound(8)`, xét điều kiện `nums[i] >= 8`:

```text
nums:        5     7     7     8     8     10
nums[i]>=8:  False False False True  True  True
                               ^
                         lower_bound = 3
```

Binary search tìm **điểm chuyển từ False sang True** — luôn tồn tại vì mảng đã sắp xếp.

## Viết `lower_bound` với đoạn nửa mở `[lo, hi)`

```python
lo, hi = 0, n          # đáp án nằm trong [0, n]
while lo < hi:
    mid = (lo + hi) // 2
    if nums[mid] < x:  # mid còn ở vùng False -> đáp án ở bên phải mid
        lo = mid + 1
    else:              # mid ở vùng True -> mid có thể là đáp án
        hi = mid
return lo
```

Khác với mục 4: không có `ans`, không có `==`. Khi `lo == hi`, đó chính là điểm chuyển.

`upper_bound` chỉ khác **một dấu**: `nums[mid] <= x`.

In [6]:
def lower_bound(nums: List[int], x: int) -> int:
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo


def upper_bound(nums: List[int], x: int) -> int:
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] <= x:
            lo = mid + 1
        else:
            hi = mid
    return lo

nums = [5, 7, 7, 8, 8, 10]
for x in [8, 7, 6, 11, 4]:
    print(f"x = {x:>2}: lower_bound = {lower_bound(nums, x)}, upper_bound = {upper_bound(nums, x)}")

x =  8: lower_bound = 3, upper_bound = 5
x =  7: lower_bound = 1, upper_bound = 3
x =  6: lower_bound = 1, upper_bound = 1
x = 11: lower_bound = 6, upper_bound = 6
x =  4: lower_bound = 0, upper_bound = 0


# 6. Module `bisect` của Python

Python có sẵn hai hàm này:

- `bisect.bisect_left(nums, x)` = `lower_bound`
- `bisect.bisect_right(nums, x)` = `upper_bound`

Nên biết để dùng trong thực tế, nhưng phỏng vấn thường muốn bạn **tự viết** được.

In [7]:
import bisect

nums = [5, 7, 7, 8, 8, 10]
print(bisect.bisect_left(nums, 8), bisect.bisect_right(nums, 8))
print(bisect.bisect_left(nums, 6), bisect.bisect_right(nums, 6))

3 5
1 1


# 7. Mô phỏng `lower_bound` và `upper_bound`

In [8]:
def trace_bound(nums: List[int], x: int, strict: bool):
    name = "upper_bound" if strict else "lower_bound"
    cond = "<=" if strict else "<"
    lo, hi = 0, len(nums)
    print(f"{name}({nums}, {x}) — điều kiện đi phải: nums[mid] {cond} {x}")
    while lo < hi:
        mid = (lo + hi) // 2
        go_right = nums[mid] <= x if strict else nums[mid] < x
        if go_right:
            print(f"  lo={lo} hi={hi} mid={mid} nums[mid]={nums[mid]:>2} -> đúng, lo = {mid + 1}")
            lo = mid + 1
        else:
            print(f"  lo={lo} hi={hi} mid={mid} nums[mid]={nums[mid]:>2} -> sai,  hi = {mid}")
            hi = mid
    print(f"  lo == hi == {lo} -> {name} = {lo}")
    return lo

nums = [5, 7, 7, 8, 8, 10]
first = trace_bound(nums, 8, strict=False)
print()
last = trace_bound(nums, 8, strict=True) - 1
print("\nĐáp án:", [first, last])

lower_bound([5, 7, 7, 8, 8, 10], 8) — điều kiện đi phải: nums[mid] < 8
  lo=0 hi=6 mid=3 nums[mid]= 8 -> sai,  hi = 3
  lo=0 hi=3 mid=1 nums[mid]= 7 -> đúng, lo = 2
  lo=2 hi=3 mid=2 nums[mid]= 7 -> đúng, lo = 3
  lo == hi == 3 -> lower_bound = 3

upper_bound([5, 7, 7, 8, 8, 10], 8) — điều kiện đi phải: nums[mid] <= 8
  lo=0 hi=6 mid=3 nums[mid]= 8 -> đúng, lo = 4
  lo=4 hi=6 mid=5 nums[mid]=10 -> sai,  hi = 5
  lo=4 hi=5 mid=4 nums[mid]= 8 -> đúng, lo = 5
  lo == hi == 5 -> upper_bound = 5

Đáp án: [3, 4]


# 8. Phiên bản đúng format LeetCode

In [9]:
class Solution:
    def searchRange(self, nums: List[int], target: int) -> List[int]:
        def lower_bound(x: int) -> int:
            lo, hi = 0, len(nums)
            while lo < hi:
                mid = (lo + hi) // 2
                if nums[mid] < x:
                    lo = mid + 1
                else:
                    hi = mid
            return lo

        first = lower_bound(target)
        if first == len(nums) or nums[first] != target:
            return [-1, -1]
        last = lower_bound(target + 1) - 1     # upper_bound(target) với số nguyên
        return [first, last]

Mẹo: với **số nguyên**, `upper_bound(target) == lower_bound(target + 1)`. Nên chỉ cần viết **một** hàm.

In [10]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<26} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", sol.searchRange([5, 7, 7, 8, 8, 10], 8), [3, 4])
check("Ví dụ 2", sol.searchRange([5, 7, 7, 8, 8, 10], 6), [-1, -1])
check("Ví dụ 3: rỗng", sol.searchRange([], 0), [-1, -1])
check("Một phần tử, có", sol.searchRange([1], 1), [0, 0])
check("Toàn bằng target", sol.searchRange([2, 2, 2, 2], 2), [0, 3])
check("Nhỏ hơn mọi phần tử", sol.searchRange([2, 3], 1), [-1, -1])
check("Lớn hơn mọi phần tử", sol.searchRange([2, 3], 4), [-1, -1])
check("Ở cuối", sol.searchRange([1, 2, 3, 3], 3), [2, 3])

Ví dụ 1                    PASS  [3, 4]
Ví dụ 2                    PASS  [-1, -1]
Ví dụ 3: rỗng              PASS  [-1, -1]
Một phần tử, có            PASS  [0, 0]
Toàn bằng target           PASS  [0, 3]
Nhỏ hơn mọi phần tử        PASS  [-1, -1]
Lớn hơn mọi phần tử        PASS  [-1, -1]
Ở cuối                     PASS  [2, 3]


In [11]:
import random

random.seed(34)
for _ in range(5000):
    nums = sorted(random.randint(-5, 5) for _ in range(random.randint(0, 15)))
    t = random.randint(-7, 7)
    expected = search_range_linear(nums, t)
    assert sol.searchRange(nums, t) == expected, (nums, t)
    assert [find_first(nums, t), find_last(nums, t)] == expected, (nums, t)
    assert lower_bound(nums, t) == bisect.bisect_left(nums, t)
    assert upper_bound(nums, t) == bisect.bisect_right(nums, t)
print("5000 test ngẫu nhiên: mọi cách khớp nhau, lower/upper_bound khớp bisect")

5000 test ngẫu nhiên: mọi cách khớp nhau, lower/upper_bound khớp bisect


# 9. Vì sao đúng? Độ phức tạp?

**Bất biến của `lower_bound`:** mọi chỉ số `< lo` có `nums[i] < x` (vùng False); mọi chỉ số `>= hi` có `nums[i] >= x` (vùng True). Ban đầu `lo = 0`, `hi = n` nên đúng hiển nhiên.

- `nums[mid] < x` → `mid` thuộc vùng False → `lo = mid + 1` giữ bất biến.
- Ngược lại → `mid` thuộc vùng True → `hi = mid` giữ bất biến.

Đoạn `[lo, hi)` thu hẹp mỗi bước (vì `lo <= mid < hi`). Khi `lo == hi`, mọi thứ bên trái là False, bên phải là True → `lo` là điểm chuyển.

```text
Time:  O(log n)  — hai lần binary search
Space: O(1)
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Dừng ngay khi gặp `target`, rồi lan ra hai bên

Đúng nhưng `O(n)` xấu nhất.

In [12]:
import time

def range_by_expanding(nums: List[int], target: int) -> List[int]:
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            l = r = mid
            while l > 0 and nums[l - 1] == target:
                l -= 1
            while r < len(nums) - 1 and nums[r + 1] == target:
                r += 1
            return [l, r]
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return [-1, -1]

big = [8] * 10**6
t = time.time(); range_by_expanding(big, 8); t1 = time.time() - t
t = time.time(); sol.searchRange(big, 8); t2 = time.time() - t
print(f"Lan ra hai bên: {t1:.4f}s   |   lower_bound: {t2:.6f}s")

Lan ra hai bên: 0.0922s   |   lower_bound: 0.000050s

## Lỗi 2: Trộn hai quy ước đoạn

`while lo < hi` đi với `hi = len(nums)` và `hi = mid`. `while lo <= hi` đi với `hi = len(nums) - 1` và `hi = mid - 1`. Trộn lẫn → lặp vô hạn hoặc sót phần tử.

In [13]:
def wrong_mixed(nums: List[int], x: int) -> int:
    lo, hi = 0, len(nums)
    steps = 0
    while lo <= hi and steps < 20:          # SAI: <= với hi = mid
        steps += 1
        mid = (lo + hi) // 2
        if mid < len(nums) and nums[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return f"dừng sau {steps} bước (bị chặn ở 20) -> lặp vô hạn nếu không chặn"

print(wrong_mixed([1, 2, 3], 2))

dừng sau 20 bước (bị chặn ở 20) -> lặp vô hạn nếu không chặn


## Lỗi 3: Quên kiểm tra `first == len(nums)`

Nếu `target` lớn hơn mọi phần tử, `lower_bound` trả `n` → `nums[n]` gây `IndexError`. Luôn kiểm tra `first < n` **trước** `nums[first] == target`.

## Lỗi 4: `upper_bound` mà không trừ 1

`upper_bound` trả vị trí **sau** phần tử cuối cùng bằng `target`. Vị trí cuối là `upper_bound - 1`.

# 11. Mẹo nhận diện pattern

Nghĩ tới **lower_bound / upper_bound** khi:

- "Vị trí **đầu tiên / cuối cùng**", "**bao nhiêu** phần tử bằng / nhỏ hơn / lớn hơn `x`" trên mảng sắp xếp.
- "Vị trí **chèn**" (**#35 Search Insert Position** chính là `lower_bound`).
- Tổng quát: tìm điểm chuyển của một điều kiện **đơn điệu** (False...False True...True) — "Binary Search on Answer" (**#875 Koko Eating Bananas**, **#1011**).

| Câu hỏi | Công thức |
|---|---|
| Số phần tử `== x` | `upper_bound(x) - lower_bound(x)` |
| Số phần tử `< x` | `lower_bound(x)` |
| Số phần tử `<= x` | `upper_bound(x)` |
| Số phần tử `> x` | `n - upper_bound(x)` |

## Template

```python
lo, hi = 0, n
while lo < hi:
    mid = (lo + hi) // 2
    if not cond(mid): lo = mid + 1     # mid ở vùng False
    else:             hi = mid         # mid ở vùng True
return lo                              # điểm chuyển đầu tiên
```

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `nums = [1, 2, 2, 2, 3, 5, 5]`, tính `lower_bound` và `upper_bound` cho `x = 2, 4, 5, 0, 9`.

## Bài 2 — Search Insert Position (LeetCode #35)

Trả về chỉ số của `target` nếu có, nếu không thì chỉ số nơi nó **sẽ được chèn** để giữ thứ tự (mảng không có trùng).

```python
search_insert([1, 3, 5, 6], 5) -> 2
search_insert([1, 3, 5, 6], 2) -> 1
search_insert([1, 3, 5, 6], 7) -> 4
```

In [14]:
def exercise_2_search_insert(nums: List[int], target: int) -> int:
    # TODO
    pass


# assert exercise_2_search_insert([1, 3, 5, 6], 5) == 2
# assert exercise_2_search_insert([1, 3, 5, 6], 2) == 1
# assert exercise_2_search_insert([1, 3, 5, 6], 7) == 4
# assert exercise_2_search_insert([1, 3, 5, 6], 0) == 0

## Bài 3 — Điền chỗ trống

In [15]:
def exercise_3_search_range(nums: List[int], target: int) -> List[int]:
    def lower_bound(x):
        lo, hi = 0, len(nums)
        # TODO 1: vòng while lo < hi, cập nhật lo / hi
        return lo

    # TODO 2: first = lower_bound(target); nếu không có target -> [-1, -1]
    # TODO 3: last = lower_bound(target + 1) - 1
    pass


# assert exercise_3_search_range([5, 7, 7, 8, 8, 10], 8) == [3, 4]
# assert exercise_3_search_range([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
# assert exercise_3_search_range([], 0) == [-1, -1]

## Bài 4 — Biến thể: đếm phần tử trong khoảng

Cho mảng sắp xếp và nhiều truy vấn `(a, b)`, mỗi truy vấn đếm số phần tử `x` với `a <= x <= b`. Mỗi truy vấn `O(log n)`.

```python
count_in_range([1, 3, 3, 5, 7, 9], [(3, 7), (0, 2), (8, 100)]) -> [4, 1, 1]
```

In [16]:
def exercise_4_count_in_range(nums: List[int], queries):
    # TODO
    pass


# assert exercise_4_count_in_range([1, 3, 3, 5, 7, 9], [(3, 7), (0, 2), (8, 100)]) == [4, 1, 1]
# assert exercise_4_count_in_range([], [(0, 1)]) == [0]

## Bài 5 — Mini challenge: Binary Search on Answer (LeetCode #875)

Koko có các đống chuối `piles` và `h` giờ. Mỗi giờ cô chọn một đống và ăn tối đa `k` quả (nếu đống ít hơn `k` thì ăn hết đống đó, giờ đó không ăn thêm). Tìm `k` **nhỏ nhất** để ăn hết trong `h` giờ.

```python
min_eating_speed([3, 6, 7, 11], 8)       -> 4
min_eating_speed([30, 11, 23, 4, 20], 5) -> 30
min_eating_speed([30, 11, 23, 4, 20], 6) -> 23
```

Gợi ý nhanh: điều kiện "ăn hết trong `h` giờ với tốc độ `k`" là **đơn điệu** theo `k` (False…False True…True). Tìm điểm chuyển như `lower_bound`, trên khoảng `k ∈ [1, max(piles)]`.

In [17]:
def exercise_5_min_eating_speed(piles: List[int], h: int) -> int:
    # TODO
    pass


# assert exercise_5_min_eating_speed([3, 6, 7, 11], 8) == 4
# assert exercise_5_min_eating_speed([30, 11, 23, 4, 20], 5) == 30
# assert exercise_5_min_eating_speed([30, 11, 23, 4, 20], 6) == 23

# 13. Gợi ý

**Bài 1**
- Mức 1: viết hàng `nums[i] >= x` dưới dạng False/True, tìm vị trí True đầu tiên.

**Bài 2**
- Mức 1: đây chính là `lower_bound(target)`.

**Bài 3**
- Mức 1: so với `Solution` mục 8.

**Bài 4**
- Mức 1: số phần tử trong `[a, b]` = `upper_bound(b) - lower_bound(a)`.

**Bài 5**
- Mức 1: số giờ cần với tốc độ `k`: `sum(ceil(p / k))`; viết `ceil` bằng `(p + k - 1) // k`.
- Mức 2: `lo, hi = 1, max(piles)`; `while lo < hi`: nếu ăn kịp với `mid` → `hi = mid`, ngược lại `lo = mid + 1`.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [18]:
nums = [1, 2, 2, 2, 3, 5, 5]
for x in [2, 4, 5, 0, 9]:
    print(f"x = {x}: lower_bound = {lower_bound(nums, x)}, upper_bound = {upper_bound(nums, x)}")

x = 2: lower_bound = 1, upper_bound = 4
x = 4: lower_bound = 5, upper_bound = 5
x = 5: lower_bound = 5, upper_bound = 7
x = 0: lower_bound = 0, upper_bound = 0
x = 9: lower_bound = 7, upper_bound = 7


## Đáp án Bài 2

In [19]:
def answer_2_search_insert(nums: List[int], target: int) -> int:
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo

assert answer_2_search_insert([1, 3, 5, 6], 5) == 2
assert answer_2_search_insert([1, 3, 5, 6], 2) == 1
assert answer_2_search_insert([1, 3, 5, 6], 7) == 4
assert answer_2_search_insert([1, 3, 5, 6], 0) == 0
print("Bài 2 PASS")

Bài 2 PASS


## Đáp án Bài 3

In [20]:
def answer_3_search_range(nums: List[int], target: int) -> List[int]:
    def lower_bound(x):
        lo, hi = 0, len(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            if nums[mid] < x:
                lo = mid + 1
            else:
                hi = mid
        return lo

    first = lower_bound(target)
    if first == len(nums) or nums[first] != target:
        return [-1, -1]
    return [first, lower_bound(target + 1) - 1]

assert answer_3_search_range([5, 7, 7, 8, 8, 10], 8) == [3, 4]
assert answer_3_search_range([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
assert answer_3_search_range([], 0) == [-1, -1]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [21]:
def answer_4_count_in_range(nums: List[int], queries):
    return [upper_bound(nums, b) - lower_bound(nums, a) for a, b in queries]

assert answer_4_count_in_range([1, 3, 3, 5, 7, 9], [(3, 7), (0, 2), (8, 100)]) == [4, 1, 1]
assert answer_4_count_in_range([], [(0, 1)]) == [0]
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [22]:
def answer_5_min_eating_speed(piles: List[int], h: int) -> int:
    def can_finish(k: int) -> bool:
        return sum((p + k - 1) // k for p in piles) <= h

    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if can_finish(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo

assert answer_5_min_eating_speed([3, 6, 7, 11], 8) == 4
assert answer_5_min_eating_speed([30, 11, 23, 4, 20], 5) == 30
assert answer_5_min_eating_speed([30, 11, 23, 4, 20], 6) == 23
print("Bài 5 PASS")

Bài 5 PASS


Cùng một template `lower_bound`, chỉ thay "mảng" bằng **khoảng các đáp án có thể** và điều kiện `nums[mid] >= x` bằng `can_finish(mid)`.

# 15. Tổng kết

> `first = lower_bound(target)`, `last = upper_bound(target) - 1`. Hai hàm này tìm **điểm chuyển** False → True của một điều kiện đơn điệu, luôn `O(log n)` kể cả khi có rất nhiều phần tử trùng.

---

## Checklist trước khi nộp

- [ ] Không lan ra hai bên sau khi tìm thấy (tránh `O(n)`)?
- [ ] Dùng **một** quy ước đoạn nhất quán (`[lo, hi)` với `lo < hi`, `hi = mid`)?
- [ ] Kiểm tra `first < n` trước khi đọc `nums[first]`?
- [ ] `last = upper_bound - 1`?
- [ ] Đã thử mảng rỗng, toàn bằng target, target ngoài khoảng giá trị?